### Algorithm from Scratch

In [1]:
import torch
import torch.nn as nn


In [56]:
# class for the backbone
from torchvision import models
class Backbone(nn.Module):
    def __init__(self, d_model: int = 256):
        super().__init__()
        resnet = models.resnet50(weights = models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        # print(self.backbone.children)
        self.projection = nn.Conv2d(2048,d_model,1)


    def forward(self,x):
        x = self.backbone(x)
        x = self.projection(x)
        # print(x.shape)
        return x

test = torch.randn((2,3,800,800))
test = Backbone()(test)
print(test.shape)

torch.Size([2, 256, 25, 25])


In [ ]:
import math
import matplotlib.pyplot as plt

class PositionalEncoding2D(nn.Module):
    def __init__(self,d_model: int = 256,
                 temperature: int = 10000):
        super().__init__()
        assert d_model % 2 == 0, "Provide the Even dimension as positional Dimension"
        self.d_model = d_model
        self.d_half = d_model // 2
        self.temperature = temperature


        i = torch.arange(self.d_half//2 , dtype=torch.float32)
        freq = self.temperature**(2*i / self.d_half)

        self.register_buffer('freq',freq)
        
    def _compute_1d_pe(self, positions: torch.Tensor) -> torch.Tensor:
        angles = positions[:, None] / self.freq[None,:]

        pe = torch.zeros(len(positions), self.d_half,
                         device= positions.device, dtype= torch.float32)
        
        pe[:, 0::2] = torch.sin(angles)
        pe[:, 1::2] = torch.cos(angles)

        return pe


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Returning only PE, combining it latter in different class iwth actuall tensor"""
        b, c, h, w = x.shape

        y_position = torch.arange(h, device=x.device, dtype=torch.float32)
        x_position = torch.arange(w, device=x.device, dtype=torch.float32)

        y_pe = self._compute_1d_pe(y_position)
        x_pe = self._compute_1d_pe(x_position)

        y_broadcast = y_pe[:, None, :].expand(h,w,self.d_half)
        x_broadcast = x_pe[None, :, :].expand(h,w,self.d_half)

        pe = torch.cat([y_broadcast,x_broadcast], dim = -1)
        pe = pe.permute(2,0,1).unsqueeze(0)

        return pe#x + pe


In [70]:
pe_module = PositionalEncoding2D(d_model=256)
dummy     = torch.zeros(1, 256, 15, 15)
output    = pe_module(dummy)    # (1, 256, 15, 15)

# Extract the PE (since input was zeros, output IS the PE)
pe = output[0]    # (256, 15, 15)

# Test 1: two different rows at same column should be different
assert not torch.allclose(pe[:, 0, 0], pe[:, 1, 0]), \
    "Row 0 and Row 1 should have different encodings"

# Test 2: two different cols at same row should be different  
assert not torch.allclose(pe[:, 0, 0], pe[:, 0, 1]), \
    "Col 0 and Col 1 should have different encodings"

# Test 3: same position across batch should be identical
dummy2   = torch.zeros(4, 256, 15, 15)
output2  = pe_module(dummy2)
assert torch.allclose(output2[0], output2[1]), \
    "Same position should get same PE regardless of batch index"

print("All assertions passed — PE is working correctly")

All assertions passed — PE is working correctly


### Encoder building

In [72]:
class EncoderLayer(nn.Module):
    def __init__(self,embedding_dim :int= 256,
                 num_heads : int = 8,
                 dropout : float = 0.1,
                #  device : str = None,
                 ffn_dim : int = 2048
                         ):
        super().__init__()
        self.mhsa = nn.MultiheadAttention(embed_dim=embedding_dim,
                                          num_heads=num_heads,
                                          dropout=dropout,
                                        #   device=device,
                                          batch_first=False)
        self.norm1 = nn.LayerNorm(normalized_shape= embedding_dim)
        self.norm2 = nn.LayerNorm(normalized_shape=embedding_dim)
        self.feedforward = nn.Linear(embedding_dim, ffn_dim)
        self.feedforward2 = nn.Linear(ffn_dim, embedding_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, pe, src_key_padding_mask = None):
        # shape : (seq, batch, dim)
        q = x + pe 
        k = x + pe
        v = x

        attention_output, _ = self.mhsa(q, k, v,
                                        need_weights = False,
                                        key_padding_mask = src_key_padding_mask) # to include key padding mask here for the q,k,v

        norm_output = self.norm1(x + self.dropout(attention_output))
        ffn_output = self.feedforward2(self.dropout(self.relu(self.feedforward(norm_output))))
        return self.norm2(ffn_output+norm_output)



In [73]:
class Encoder(nn.Module):
    def __init__(self,
                 heads:int = 8,
                 dff : int = 2048,
                 dropout : float = 0.1,
                 embedding_dim : int = 256,
                 num_encoders : int = 6
                 ):
        super().__init__()
        # self.encoder_layer = EncoderLayer(embedding_dim=embedding_dim,
        #                                   num_heads=heads,
        #                                   dropout=dropout,
        #                                   ffn_dim=dff)
        self.encoder = nn.ModuleList([EncoderLayer(embedding_dim=embedding_dim,
                                          num_heads=heads,
                                          dropout=dropout,
                                          ffn_dim=dff) for _ in range(num_encoders)])

    def forward(self, x, pe, src_key_padding = None):
        for encoder_layer in self.encoder:
            x = encoder_layer(x, pe, src_key_padding)
        
        return x


In [74]:
enc = Encoder()
ids = [id(layer) for layer in enc.encoder]
print(len(set(ids)))  # should print 6, not 1

6


### Building decoder layer and decoder

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self,):
        super().__init__()
        